# B1 · Ejercicios de NumPy: del bucle al arreglo

**Nivel Intermedio** · Trabajo práctico B1 · Semana 2 · Se entrega al inicio de la clase de la semana 3 · Tiempo estimado: 3–4 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/B1_ejercicios_numpy.ipynb)

Seis funciones que aparecen en todo proyecto de *machine learning*, escritas con bucles de Python. Tu trabajo: reescribir cada una **sin bucles**, con operaciones de NumPy sobre arreglos enteros (**vectorizar**, *vectorization*), comprobar que da lo mismo y medir cuánto más rápida es.

**Objetivos.** Al terminar podrás:

1. Traducir un bucle a operaciones de NumPy usando `axis`, `keepdims`, *broadcasting* e indexación con arreglos.
2. Comprobar una implementación nueva contra una de referencia en la que confías.
3. Medir tiempos de ejecución con cuidado y reportar una aceleración (*speedup*) con un gráfico.

**Prerrequisitos.** Clases de Python (semana 1) y NumPy (semana 2: formas, `axis`, *broadcasting*, `@`). Si faltaste a esas clases, la sección "Repaso rápido" de abajo trae lo necesario.

**Cómo entregar.** Antes de escribir nada: *Archivo → Guardar una copia en GitHub*, repositorio `Alto-Rendimiento-IA/entregas-<tu-usuario>`, rama `main`, ruta **`B1/B1_ejercicios_numpy.ipynb`**, mensaje `Empezar B1`. Trabaja en la copia que se abre y guarda ahí después de cada ejercicio. Lo que esté en tu repositorio al inicio de la clase de la semana 3 es tu entrega.

**Uso de IA.** Puedes preguntarle a una IA qué hace `keepdims` o por qué un *broadcasting* falla. No le pidas la versión vectorizada de las funciones: ese es el ejercicio.

## Preparación

Ejecuta esta celda primero. Carga las librerías, fija la semilla y define dos ayudas:

- `no_loops(f)`: revisa que el código de `f` no tenga `for` ni `while` (tampoco listas por comprensión, que son bucles disfrazados).
- `time_it(f, *args)`: ejecuta `f(*args)` varias veces y devuelve el mejor tiempo, en segundos.

In [ ]:
import ast
import inspect
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

SEED = 0
rng = np.random.default_rng(SEED)


def no_loops(f):
    """Falla si f contiene un bucle (for, while o una lista por comprensión)."""
    tree = ast.parse(textwrap.dedent(inspect.getsource(f)))
    loops = (ast.For, ast.While, ast.comprehension)
    assert not any(isinstance(node, loops) for node in ast.walk(tree)), f"{f.__name__} todavía usa un bucle"


def time_it(f, *args, repeat=3):
    """El mejor de `repeat` tiempos de f(*args), en segundos."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        f(*args)
        best = min(best, time.perf_counter() - start)
    return best


digits = load_digits()
images = digits.images  # (1797, 8, 8): imágenes de dígitos escritos a mano, valores de 0 a 16
X = digits.data         # (1797, 64): las mismas imágenes, aplanadas en filas
y = digits.target       # (1797,): el dígito de cada imagen
print("images:", images.shape, " X:", X.shape, " y:", y.shape)

## Repaso rápido

Lo que vas a necesitar, con un ejemplo de cada cosa. Ejecuta la celda y mira las formas.

- **Reducciones con `axis`**: el eje que nombras desaparece. `A.sum(axis=0)` suma por columnas; `A.max(axis=(1, 2))` reduce dos ejes a la vez.
- **`keepdims=True`**: el eje reducido queda con tamaño 1 en lugar de desaparecer. Así el resultado se puede combinar por *broadcasting* con el arreglo original.
- ***Broadcasting***: dos formas son compatibles, comparadas desde la derecha, si en cada eje los tamaños son iguales o uno es 1. `(5, 3) - (3,)` resta una fila a cada fila; `(5, 3) - (5, 1)` resta una columna a cada columna.
- **Indexación con arreglos** (*fancy indexing*): `M[filas, columnas]` con dos arreglos del mismo largo elige los elementos `M[filas[0], columnas[0]]`, `M[filas[1], columnas[1]]`, …
- **`np.where(cond, a, b)`**: elige elemento a elemento `a` donde `cond` es `True` y `b` donde no.
- **`@`**: producto de matrices, `(n, d) @ (d, m) → (n, m)`.

In [ ]:
A = np.arange(15).reshape(5, 3)
print("A.sum(axis=0):", A.sum(axis=0), " forma", A.sum(axis=0).shape)
print("A.max(axis=1, keepdims=True) tiene forma", A.max(axis=1, keepdims=True).shape)
print("A - A.max(axis=1, keepdims=True):")
print(A - A.max(axis=1, keepdims=True))

M = np.zeros((3, 4))
M[np.array([0, 1, 2]), np.array([3, 0, 1])] = 1  # pone un 1 en (0, 3), (1, 0) y (2, 1)
print(M)
print(np.where(np.array([1, -2, 3]) > 0, 10, -10))

**Cómo está organizado cada ejercicio.** Primero viene la versión con bucles, `<nombre>_loop`: **no la cambies**, es tu referencia (y tu línea base de tiempo). Después escribes `<nombre>_vec`, que debe dar el mismo resultado sin ningún bucle. La celda de verificación compara las dos con datos aleatorios y casos borde.

Empieza siempre igual: escribe en un comentario **la forma de cada arreglo intermedio**. Casi todos los errores de NumPy son formas equivocadas.

## Ejercicio 1: normalizar imágenes

Para mostrar o comparar imágenes, conviene llevar cada una al rango $[0, 1]$: a cada píxel se le resta el mínimo **de su imagen** y se divide por (máximo − mínimo) **de su imagen**. Si una imagen es constante (máximo = mínimo), el resultado es todo ceros.

`images` tiene forma `(n, h, w)`. Pista: `min` y `max` con `axis=(1, 2)` y `keepdims=True` dan forma `(n, 1, 1)`. Para no dividir por cero, usa `np.where` para reemplazar los rangos nulos por 1 antes de dividir.

In [ ]:
def normalize_images_loop(images):
    n, h, w = images.shape
    out = np.zeros((n, h, w))
    for i in range(n):
        lo = images[i, 0, 0]
        hi = images[i, 0, 0]
        for r in range(h):
            for c in range(w):
                lo = min(lo, images[i, r, c])
                hi = max(hi, images[i, r, c])
        if hi > lo:
            for r in range(h):
                for c in range(w):
                    out[i, r, c] = (images[i, r, c] - lo) / (hi - lo)
    return out

In [ ]:
def normalize_images_vec(images):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(normalize_images_vec)
imgs = rng.normal(size=(20, 5, 7))
imgs[3] = 2.5  # una imagen constante
out = normalize_images_vec(imgs)
assert out.shape == imgs.shape, f"forma {out.shape}, esperaba {imgs.shape}"
assert np.allclose(out, normalize_images_loop(imgs)), "no coincide con normalize_images_loop"
assert np.allclose(out[3], 0), "una imagen constante debe quedar en ceros"
assert np.allclose(normalize_images_vec(images[:50]), normalize_images_loop(images[:50])), "falla con los dígitos"
print("✓ Ejercicio 1")

**Guarda en GitHub** con el mensaje `Ejercicio 1 resuelto`.

## Ejercicio 2: estandarizar columnas

Antes de entrenar casi siempre **estandarizamos** (*standardize*) cada variable (cada columna de `X`): le restamos su media y la dividimos por su desviación estándar. Las columnas constantes tienen desviación 0; para ellas se divide por 1.

La desviación estándar es la de población, $\sigma = \sqrt{\frac{1}{n}\sum_i (x_i - \mu)^2}$, que es lo que calcula `np.std` por defecto. Pista: `X.mean(axis=0)` tiene forma `(d,)`, y `(n, d) - (d,)` funciona por *broadcasting*.

In [ ]:
def standardize_loop(X):
    n, d = X.shape
    out = np.zeros((n, d))
    for j in range(d):
        mean = 0.0
        for i in range(n):
            mean += X[i, j]
        mean /= n
        var = 0.0
        for i in range(n):
            var += (X[i, j] - mean) ** 2
        std = (var / n) ** 0.5
        if std == 0:
            std = 1.0
        for i in range(n):
            out[i, j] = (X[i, j] - mean) / std
    return out

In [ ]:
def standardize_vec(X):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(standardize_vec)
A = rng.normal(loc=5, scale=3, size=(40, 6))
A[:, 2] = 7.0  # una columna constante
out = standardize_vec(A)
assert out.shape == A.shape, f"forma {out.shape}, esperaba {A.shape}"
assert np.allclose(out, standardize_loop(A)), "no coincide con standardize_loop"
assert np.allclose(out[:, 2], 0), "una columna constante debe quedar en ceros"
assert np.allclose(standardize_vec(X), standardize_loop(X)), "falla con los dígitos"
print("✓ Ejercicio 2")

**Guarda en GitHub** con el mensaje `Ejercicio 2 resuelto`.

## Ejercicio 3: codificación *one-hot*

Una red que clasifica en $k$ clases produce $k$ números por ejemplo, y para compararlos con la respuesta correcta conviene escribir la etiqueta también como $k$ números: un 1 en la posición de la clase y 0 en las demás. Eso es la **codificación *one-hot***. Con $k = 4$, la etiqueta 2 se escribe `[0, 0, 1, 0]`.

`one_hot_vec(y, k)` recibe `y` de forma `(n,)` con enteros entre 0 y `k − 1`, y devuelve una matriz de forma `(n, k)`. Dos caminos, elige uno:

- Crear una matriz de ceros y poner los unos con indexación con arreglos: las filas son `np.arange(n)`, las columnas son `y`.
- Comparar con *broadcasting*: `y[:, None]` tiene forma `(n, 1)` y `np.arange(k)` forma `(k,)`.

In [ ]:
def one_hot_loop(y, k):
    n = len(y)
    out = np.zeros((n, k))
    for i in range(n):
        out[i, y[i]] = 1.0
    return out

In [ ]:
def one_hot_vec(y, k):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(one_hot_vec)
assert np.array_equal(one_hot_vec(np.array([2, 0]), 4), [[0, 0, 1, 0], [1, 0, 0, 0]]), "revisa el ejemplo pequeño"
labels = rng.integers(0, 7, size=100)
out = one_hot_vec(labels, 7)
assert out.shape == (100, 7), f"forma {out.shape}, esperaba (100, 7)"
assert np.array_equal(out, one_hot_loop(labels, 7)), "no coincide con one_hot_loop"
assert one_hot_vec(np.array([0, 0]), 5).shape == (2, 5), "k fija las columnas aunque falten clases en y"
print("✓ Ejercicio 3")

**Guarda en GitHub** con el mensaje `Ejercicio 3 resuelto`.

## Ejercicio 4: *softmax* por filas

La función ***softmax*** convierte una fila de puntajes cualesquiera en probabilidades: números positivos que suman 1. La veremos a fondo en la semana 4; hoy solo importa la cuenta. Para una fila $z$:

$$\text{softmax}(z)_j = \frac{e^{z_j - m}}{\sum_l e^{z_l - m}}, \qquad m = \max_l z_l$$

Restar el máximo $m$ no cambia el resultado (se cancela arriba y abajo), pero evita que `np.exp` se desborde: $e^{1000}$ es infinito en la computadora, $e^{0}$ no.

`softmax_vec(Z)` recibe `Z` de forma `(n, k)` y aplica la *softmax* a cada fila. Pista: `keepdims=True` en el `max` y en la `sum`.

In [ ]:
import math


def softmax_loop(Z):
    n, k = Z.shape
    out = np.zeros((n, k))
    for i in range(n):
        m = max(Z[i])
        total = 0.0
        for j in range(k):
            out[i, j] = math.exp(Z[i, j] - m)
            total += out[i, j]
        for j in range(k):
            out[i, j] /= total
    return out

In [ ]:
def softmax_vec(Z):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(softmax_vec)
Z = rng.normal(scale=3, size=(30, 5))
out = softmax_vec(Z)
assert out.shape == Z.shape, f"forma {out.shape}, esperaba {Z.shape}"
assert np.allclose(out, softmax_loop(Z)), "no coincide con softmax_loop"
assert np.allclose(out.sum(axis=1), 1), "cada fila debe sumar 1"
big = softmax_vec(np.array([[1000.0, 1001.0], [-1000.0, -1000.0]]))
assert np.all(np.isfinite(big)), "con puntajes grandes aparecen nan o inf: ¿restaste el máximo de cada fila?"
assert np.allclose(big, softmax_loop(np.array([[1000.0, 1001.0], [-1000.0, -1000.0]]))), "falla con puntajes grandes"
print("✓ Ejercicio 4")

**Guarda en GitHub** con el mensaje `Ejercicio 4 resuelto`.

## Ejercicio 5: similitud coseno entre todos los pares

La **similitud coseno** (*cosine similarity*) mide cuánto apuntan dos vectores en la misma dirección, sin importar su largo:

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Vale 1 si apuntan igual, 0 si son perpendiculares. Se usa todo el tiempo para comparar *embeddings* (semana 20 en adelante).

`cosine_similarity_vec(A, B)` recibe `A` de forma `(n, d)` y `B` de forma `(m, d)`, sin filas de ceros, y devuelve `S` de forma `(n, m)` con `S[i, j] = cos(A[i], B[j])`. Plan en dos pasos:

1. Divide cada fila de `A` por su norma, y lo mismo con `B`. Pista: `np.linalg.norm(A, axis=1, keepdims=True)`.
2. Con las filas ya de norma 1, el coseno es solo el producto punto: un único producto de matrices.

In [ ]:
def cosine_similarity_loop(A, B):
    n, d = A.shape
    m = B.shape[0]
    out = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            dot = 0.0
            norm_a = 0.0
            norm_b = 0.0
            for t in range(d):
                dot += A[i, t] * B[j, t]
                norm_a += A[i, t] ** 2
                norm_b += B[j, t] ** 2
            out[i, j] = dot / (norm_a**0.5 * norm_b**0.5)
    return out

In [ ]:
def cosine_similarity_vec(A, B):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(cosine_similarity_vec)
P = rng.normal(size=(8, 5))
Q = rng.normal(size=(11, 5))
S = cosine_similarity_vec(P, Q)
assert S.shape == (8, 11), f"forma {S.shape}, esperaba (8, 11)"
assert np.allclose(S, cosine_similarity_loop(P, Q)), "no coincide con cosine_similarity_loop"
assert np.allclose(np.diag(cosine_similarity_vec(P, P)), 1), "un vector consigo mismo tiene similitud 1"
assert np.allclose(cosine_similarity_vec(P, 3 * P), cosine_similarity_vec(P, P)), "el largo no debe importar"
print("✓ Ejercicio 5")

**Guarda en GitHub** con el mensaje `Ejercicio 5 resuelto`.

## Ejercicio 6: matriz de confusión

La ***accuracy*** dice cuántas veces acierta un clasificador, pero no **en qué se equivoca**. La **matriz de confusión** (*confusion matrix*) sí: `C[t, p]` cuenta los ejemplos de la clase verdadera `t` que el modelo predijo como `p`. La diagonal son los aciertos; cada casilla fuera de la diagonal es un tipo de error ("confundió un 3 con un 8").

`confusion_matrix_vec(y_true, y_pred, k)` devuelve una matriz de enteros de forma `(k, k)`. Es el ejercicio más difícil: con `C[y_true, y_pred] += 1` **no** funciona, porque si un par `(t, p)` se repite NumPy lo suma una sola vez. Dos salidas, elige una:

- `np.add.at(C, (y_true, y_pred), 1)`: suma sin perder repeticiones.
- Un truco más elegante: el par `(t, p)` se puede escribir como un solo número, `t * k + p`, entre 0 y `k² − 1`. Cuenta cuántas veces aparece cada número con `np.bincount(..., minlength=k * k)` y dale forma `(k, k)`.

In [ ]:
def confusion_matrix_loop(y_true, y_pred, k):
    C = np.zeros((k, k), dtype=int)
    for i in range(len(y_true)):
        C[y_true[i], y_pred[i]] += 1
    return C

In [ ]:
def confusion_matrix_vec(y_true, y_pred, k):
    # TODO
    raise NotImplementedError

In [ ]:
no_loops(confusion_matrix_vec)
t = np.array([0, 0, 1, 2, 2, 2])
p = np.array([0, 1, 1, 2, 2, 0])
assert np.array_equal(confusion_matrix_vec(t, p, 3), [[1, 1, 0], [0, 1, 0], [1, 0, 2]]), "revisa el ejemplo pequeño (¿se pierden los pares repetidos?)"
t = rng.integers(0, 10, size=500)
p = rng.integers(0, 10, size=500)
C = confusion_matrix_vec(t, p, 10)
assert C.shape == (10, 10), f"forma {C.shape}, esperaba (10, 10)"
assert np.array_equal(C, confusion_matrix_loop(t, p, 10)), "no coincide con confusion_matrix_loop"
assert C.sum() == 500, "la suma de la matriz debe ser la cantidad de ejemplos"
print("✓ Ejercicio 6")

**Guarda en GitHub** con el mensaje `Ejercicio 6 resuelto`.

## Ejercicio 7: medir la aceleración

Ahora la pregunta del trabajo: **¿cuánto más rápidas son tus versiones?** La versión con bucles es la línea base. La aceleración es

$$\text{aceleración} = \frac{\text{tiempo con bucles}}{\text{tiempo vectorizado}}$$

Una aceleración de 50 significa que la versión vectorizada tarda la quincuagésima parte.

Algunos cuidados al medir tiempos, que ya resuelve `time_it`: se mide varias veces y se toma el **mejor** tiempo (los demás incluyen interrupciones de la computadora), y se usan datos del mismo tamaño para las dos versiones.

La celda de abajo prepara los datos de cada función, con tamaños elegidos para que la versión con bucles tarde menos de un segundo. Ejecútala sin cambiarla.

In [ ]:
pred_digits = rng.integers(0, 10, size=20_000)  # predicciones al azar, solo para medir tiempos
benchmarks = {
    "normalize_images": (normalize_images_loop, normalize_images_vec, (images[:400],)),
    "standardize": (standardize_loop, standardize_vec, (X[:600],)),
    "one_hot": (one_hot_loop, one_hot_vec, (rng.integers(0, 10, size=50_000), 10)),
    "softmax": (softmax_loop, softmax_vec, (rng.normal(size=(5_000, 10)),)),
    "cosine_similarity": (cosine_similarity_loop, cosine_similarity_vec, (X[:60], X[60:110])),
    "confusion_matrix": (confusion_matrix_loop, confusion_matrix_vec, (rng.integers(0, 10, size=20_000), pred_digits, 10)),
}

### Tu tarea

**7a.** Para cada entrada de `benchmarks`, mide con `time_it` el tiempo de la versión con bucles y el de la vectorizada (los dos con los mismos argumentos: `time_it(f, *args)`) y agrega a `results` una tupla `(nombre, tiempo_bucle, tiempo_vec)`. Aquí sí puedes usar un bucle: recorre `benchmarks.items()`.

Después ejecuta la celda siguiente: imprime la tabla y dibuja el gráfico.

In [ ]:
results = []
# TODO 7a

In [ ]:
assert len(results) == 6, "results debe tener una tupla por función (6)"
speedups = {name: t_loop / t_vec for name, t_loop, t_vec in results}

print(f"{'función':<20}{'bucles (ms)':>14}{'vectorizada (ms)':>18}{'aceleración':>14}")
for name, t_loop, t_vec in results:
    print(f"{name:<20}{t_loop * 1000:>14.2f}{t_vec * 1000:>18.3f}{speedups[name]:>13.0f}×")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(list(speedups), list(speedups.values()), color="tab:blue")
ax.set_xscale("log")
ax.set_xlim(left=1)  # 1× = igual de rápido que el bucle
ax.set_xlabel("aceleración (veces más rápido, escala logarítmica)")
ax.set_title("Vectorizada vs bucles de Python")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**7b. ¿La aceleración depende del tamaño?** Mide `softmax_loop` y `softmax_vec` con matrices `rng.normal(size=(n, 10))` para `n` en `sizes`, y guarda la aceleración de cada tamaño en la lista `softmax_speedups` (en el mismo orden). Después ejecuta el gráfico.

In [ ]:
sizes = [10, 100, 1_000, 10_000]
softmax_speedups = []
# TODO 7b

In [ ]:
assert len(softmax_speedups) == len(sizes), "una aceleración por tamaño"
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sizes, softmax_speedups, "o-")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("filas de Z (n)")
ax.set_ylabel("aceleración")
ax.set_title("Aceleración de softmax según el tamaño")
plt.tight_layout()
plt.show()
print([f"{s:.0f}×" for s in softmax_speedups])

**Guarda en GitHub** con el mensaje `Ejercicio 7 resuelto`.

## Ejercicio 8: todo junto, un clasificador de dígitos

Tus funciones alcanzan para un clasificador completo, sin entrenar nada. Para cada dígito calculamos su **imagen promedio** con los datos de entrenamiento, y cada imagen nueva se clasifica como el dígito cuyo promedio es **más parecido** según tu similitud coseno. La matriz de confusión dice en qué se equivoca. La línea base es predecir siempre el dígito más común.

Ejecuta la celda; no tienes que cambiar nada.

In [ ]:
perm = rng.permutation(len(X))
train_idx, test_idx = perm[:1400], perm[1400:]
X_train, y_train, X_test, y_test = X[train_idx], y[train_idx], X[test_idx], y[test_idx]

majority = np.bincount(y_train).argmax()
baseline_acc = np.mean(y_test == majority)

class_means = one_hot_vec(y_train, 10).T @ X_train / np.bincount(y_train)[:, None]  # (10, 64)
similarity = cosine_similarity_vec(X_test, class_means)                              # (397, 10)
probs = softmax_vec(20 * similarity)                                                 # (397, 10)
pred = probs.argmax(axis=1)
acc = np.mean(pred == y_test)
C = confusion_matrix_vec(y_test, pred, 10)

print(f"línea base (siempre {majority}):          accuracy {baseline_acc:.3f}")
print(f"promedio más parecido (coseno):  accuracy {acc:.3f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(C, cmap="Blues")
for t in range(10):
    for p in range(10):
        ax.text(p, t, C[t, p], ha="center", va="center", fontsize=8, color="white" if C[t, p] > C.max() / 2 else "black")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("dígito predicho")
ax.set_ylabel("dígito verdadero")
ax.set_title("Matriz de confusión (prueba)")
plt.tight_layout()
plt.show()

**8a.** Lee la celda anterior con calma. La línea `class_means = one_hot_vec(y_train, 10).T @ X_train / ...` calcula los 10 promedios con un solo producto de matrices. Explica en dos o tres oraciones por qué funciona: ¿qué forma tiene cada factor, y qué suma la fila `c` del producto?

**8b.** Según la matriz de confusión, ¿cuáles son las dos confusiones más frecuentes (casillas fuera de la diagonal)? ¿Tienen sentido mirando cómo se escriben esos dígitos?

**8c.** La *softmax* no cambia cuál es la clase predicha (el `argmax` de `probs` y de `similarity` es el mismo). ¿Por qué?

*Tus respuestas:*

- 8a.
- 8b.
- 8c.

## Ejercicio 9: interpretar las mediciones

Responde con tus números:

1. ¿Qué función se aceleró más y cuál menos? Mira cuántas operaciones hace cada bucle por elemento y propón una explicación.
2. Según el ejercicio 7b, ¿qué pasa con la aceleración cuando los datos son muy chicos? ¿Por qué crees que pasa?
3. Una red de la semana 10 procesa unas 60 000 imágenes por época. Si una función tardara 50 ms con NumPy, ¿cuánto tardaría con bucles, usando la aceleración que mediste para una función parecida? ¿Qué significa eso para entrenar 20 épocas?

*Tus respuestas:*

1.
2.
3.

**Guarda en GitHub** con el mensaje `Ejercicios 8 y 9 resueltos`.

## Conclusión

*Escribe aquí un párrafo en lenguaje sencillo: qué aprendiste sobre vectorizar, qué significan las aceleraciones que mediste (comparadas con la línea base de bucles) y qué te gustaría probar después. Sin este párrafo el trabajo no tiene nota.*

## Uso de IA

*Una línea: para qué usaste IA en este trabajo, o "No usé IA".*

## Rúbrica

Primero asígnate tu puntaje (columna "Yo"). Tu compañero revisor completa la columna "Revisor" con un comentario en la copia de Drive que le compartas.

| Criterio | Qué se mira | Yo (0–2) | Revisor (0–2) |
|---|---|---|---|
| **Se ejecuta** | *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores | | |
| **Es correcto** | Las seis verificaciones imprimen ✓; las versiones vectorizadas no tienen bucles; las mediciones del ejercicio 7 están completas | | |
| **Está justificado** | Las respuestas de los ejercicios 8 y 9 explican el porqué con tus números | | |
| **Está bien comunicado** | La conclusión se entiende sin leer el código; hay un *commit* por ejercicio con un mensaje claro | | |
| **Total** | | /8 | /8 |

**Antes de la clase de la semana 3**: ejecuta *Entorno de ejecución → Reiniciar y ejecutar todo*, revisa que todo imprima ✓ y guarda por última vez en `B1/B1_ejercicios_numpy.ipynb`.